### 프로세서별 토큰 효율 분석
- 파일 : nA3_token_bench.ipynb
- 역할 : 공개 벤치마크 데이터로 이론 상한 대비 달성률과 토큰당 에너지를 계산합니다.
- 기능 :
    1. 벤치마크 데이터 불러오기  
    2. 이론 상한과 달성률  
    3. 토큰당 에너지 계산  
    4. 토큰당 에너지 비교 그래프  


# 3장 프로세서별 토큰 효율: 공개 벤치마크 데이터 분석

부록 원고의 코드와 동일합니다. 핵심 수식은 코드의 주석에 함께 적어 두었습니다.

>> [코드 A3-1] 벤치마크 데이터 불러오기 (nA3_token_bench.ipynb, STEP 1)
<hr>

In [ ]:
#- 모듈로딩
import pandas as pd

#- 공개 벤치마크 CSV를 저장소에서 직접 로딩
DATA_FILE = "data/bench/token_bench_public.csv"
CSV_URL = f"https://raw.githubusercontent.com/npu-ondevice-ai/npu-course/main/{DATA_FILE}"
df = pd.read_csv(CSV_URL)
print(df[["platform", "model", "tokens_per_sec", "power_w"]])

# 확인 포인트 -------------------------------------------------------------------------
# - 행 5개 체크, power_w가 NaN인 장비 확인

>> [코드 A3-2] 이론 상한과 달성률 (STEP 2)
<hr>

In [ ]:
#- 이론 상한과 달성률
# 상한(tokens/sec) = 대역폭(bandwidth_gbs) / 가중치 크기(weights_gb)
df["ceiling_tps"] = (df["bandwidth_gbs"] / df["weights_gb"]).round(1)
df["achieved_pct"] = (df["tokens_per_sec"] / df["ceiling_tps"] * 100).round(1)
print(df[["platform", "quantization", "ceiling_tps", "achieved_pct"]])

# 확인 포인트 --------------------------------------------------------------------------
# - ceiling_tps와 achieved_pct 열 출력 체크
# - 달성률 100% 이하 확인

**실행 결과 예시**
```
                platform quantization  ceiling_tps  achieved_pct
0   Raspberry Pi 5 (CPU)         Q6_K         13.9          77.3
1   Raspberry Pi 5 (CPU)          F16          5.7          75.8
2  RTX 4050 Laptop (GPU)         4bit          NaN           NaN
3        Hailo-10H (NPU)         4bit          NaN           NaN
4        Hailo-10H (NPU)         4bit          NaN           NaN
```

>> [코드 A3-3] 토큰당 에너지 계산 (STEP 3)
<hr>

In [ ]:
#- 토큰당 에너지
# 토큰당 에너지(mJ) = 전력(power_w) / 초당 토큰(tokens_per_sec) × 1000
df["mj_per_token"] = (df["power_w"] / df["tokens_per_sec"] * 1000).round(0)
print(df[["platform", "model", "mj_per_token"]])

# 확인 포인트 --------------------------------------------------------------------------
# - mj_per_token 출력 체크 (power_w가 NaN인 행은 NaN)

**실행 결과 예시**
```
                platform         model  mj_per_token
0   Raspberry Pi 5 (CPU)  Llama 3.2 1B           NaN
1   Raspberry Pi 5 (CPU)  Llama 3.2 1B           NaN
2  RTX 4050 Laptop (GPU)  Qwen2.5 1.5B         259.0
3        Hailo-10H (NPU)  Qwen2.5 1.5B         270.0
4        Hailo-10H (NPU)    Llama 2 7B         500.0
```

>> [코드 A3-4] 토큰당 에너지 비교 그래프 (STEP 4)
<hr>

In [ ]:
#- 모듈로딩
import matplotlib.pyplot as plt

#- 전력값이 있는 장비만 막대그래프로
sub = df.dropna(subset=["mj_per_token"])
labels = sub["platform"] + "\n" + sub["model"]
plt.figure(figsize=(7, 4))
plt.bar(labels, sub["mj_per_token"])
plt.ylabel("mJ / token")
plt.title("Energy per token (public benchmarks)")
plt.tight_layout()
plt.show()

# 확인 포인트 --------------------------------------------------------------------------
# - 속도 19배 차이 두 장비의 막대 높이가 비슷한 이유 정리
# - 같은 NPU에서 7B 모델의 막대가 1.5B급보다 높은 이유 정리
# - 표에 행 추가 시 반드시 필요한 출처 정보 정리